# 🍜 Yumify — Hệ thống AI Gợi ý Ẩm thực & Tối ưu hóa Ngân sách

**Môn học:** Nhập môn Trí tuệ Nhân tạo (Introduction to AI)  
**Mã môn học:** CO3061  
**Học kỳ:** II — Năm học 2025–2026  
**Trường:** Đại học Bách Khoa, ĐHQG-HCM  
**GVHD:** TS. Trương Vĩnh Lân  

| Họ và tên | MSSV | Email |
|-----------|------|-------|
| Đinh Đoàn Vy | 2353350 | vy.dinhdoan2005@hcmut.edu.vn |
| Trần Thiên Lộc | 2352715 | |
| Nguyễn Trần Gia Bảo | 2352103 | |
| Dương Lê Nhật Duy | 2352171 | |

**GitHub:** [https://github.com/dvydinh/yumify](https://github.com/dvydinh/yumify)  
**Báo cáo PDF:** `reports/report.pdf`

---

### Pipeline AI tổng thể
```
User Input → NLP Parser (NER) → ML Naive Bayes → Knowledge Base → CSP Solver → A* Search → Bayesian Risk → Output
```

### Các trụ cột AI
| # | Trụ cột | Module | Kỹ thuật cốt lõi |
|---|---------|--------|------------------|
| 1 | Học máy (ML) | `nlp_parser.py` + `ml_classifier.py` | Multinomial Naive Bayes (trained on recipes.json) |
| 2 | Ràng buộc (CSP) | `csp_solver.py` | Dual-constraint Pruning + Bounds Propagation |
| 3 | Logic | `knowledge_base.py` | Forward Chaining (Horn Clause) |
| 4 | Tìm kiếm (Search) | `search_engine.py` | A* + Canonical State Representation |
| 5 | Xác suất (Bayes) | `bayes_risk.py` | Extended Noisy-OR + Synergy Hidden Causes |

---
## Cell 1: Cài đặt môi trường & Clone mã nguồn

- Clone repository từ GitHub (nguồn công khai)
- Cài đặt thư viện
- **Không** mount Google Drive hay bất kỳ cloud cá nhân nào

In [ ]:
import os, sys

# ============================================================
# 1. Clone mã nguồn từ GitHub (nguồn công khai)
# ============================================================
REPO_URL = "https://github.com/dvydinh/yumify.git"
PROJECT_DIR = "/content/yumify"

if not os.path.exists(PROJECT_DIR):
    print("📦 Đang clone repository từ GitHub...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("📦 Repository đã tồn tại, đang cập nhật...")
    !cd {PROJECT_DIR} && git pull origin main

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"\n📂 Working directory: {os.getcwd()}")
print(f"📂 Nội dung: {os.listdir('.')}")

# ============================================================
# 2. Cài đặt thư viện bổ sung
# ============================================================
print("\n📚 Cài đặt thư viện...")
!pip install -q numpy requests

print("\n✅ Môi trường sẵn sàng.")

---
## Cell 2: Tải dữ liệu & Trích xuất đặc trưng TF-IDF

- Dữ liệu `recipes.json`, `ingredients.json`, `rules.json` đã có sẵn trong repo
- TF-IDF features được trích xuất tại runtime và lưu vào `data/tfidf_features.npy`

In [ ]:
import json
import numpy as np

# ============================================================
# 1. Load dữ liệu cục bộ
# ============================================================
data_dir = os.path.join(PROJECT_DIR, 'data')

with open(os.path.join(data_dir, 'recipes.json'), 'r', encoding='utf-8') as f:
    recipes = json.load(f)
with open(os.path.join(data_dir, 'ingredients.json'), 'r', encoding='utf-8') as f:
    ingredients_db = json.load(f)

print(f"📊 Dataset:")
print(f"   Recipes: {len(recipes)} công thức")
print(f"   Ingredients DB: {len(ingredients_db)} nguyên liệu")

# Thống kê cuisines
from collections import Counter
cuisine_counts = Counter(r.get('cuisine', 'N/A') for r in recipes)
print(f"\n🍳 Phân bố Cuisine:")
for cuisine, count in cuisine_counts.most_common():
    print(f"   {cuisine}: {count} recipes")

# ============================================================
# 2. Trích xuất TF-IDF Features & lưu .npy
# ============================================================
from features.feature_extractor import RecipeFeatureExtractor

extractor = RecipeFeatureExtractor()
extractor.fit(recipes)

tfidf_path = os.path.join(data_dir, 'tfidf_features')
extractor.save_embeddings(tfidf_path)

print(f"\n✅ TF-IDF features extracted: {len(extractor.vocabulary)} terms, {len(extractor.tfidf_matrix)} docs")
print(f"✅ Saved: {tfidf_path}.npy")

---
## Cell 3: Khởi tạo 5 Module AI

| Module | Class | Chức năng |
|--------|-------|-----------|
| NLP Parser | `EnglishNLPParser` | NER + ML Naive Bayes tích hợp |
| CSP Solver | `IngredientCSPSolver` | Dual-constraint Pruning |
| A* Search | `AStarMealPlanner` | Canonical State Space |
| Knowledge Base | `ForwardChainingEngine` | Forward Chaining (Horn Clause) |
| Bayesian Risk | `BayesianRecipeEvaluator` | Extended Noisy-OR + Synergy |

In [ ]:
# ============================================================
# Import tất cả module AI
# ============================================================
from modules.nlp_parser import EnglishNLPParser
from modules.csp_solver import IngredientCSPSolver
from modules.search_engine import AStarMealPlanner
from modules.knowledge_base import ForwardChainingEngine
from modules.bayes_risk import BayesianRecipeEvaluator
from modules.ml_classifier import CuisineNaiveBayesClassifier

# ============================================================
# Khởi tạo
# ============================================================
print("🧠 NLP Parser (tích hợp ML Naive Bayes)...")
nlp_parser = EnglishNLPParser()

print("🔧 CSP Solver (Dual-constraint Pruning)...")
csp_solver = IngredientCSPSolver()

print("🔍 A* Search (Canonical State Representation)...")
search_engine = AStarMealPlanner()

print("📖 Knowledge Base (Forward Chaining)...")
rules_path = os.path.join(data_dir, 'rules.json')
kb_engine = ForwardChainingEngine(rules_path)

print("📊 Bayesian Risk (Extended Noisy-OR + Synergy)...")
bayes_eval = BayesianRecipeEvaluator()

print("\n✅ Tất cả 5 trụ cột AI đã sẵn sàng!")

---
## Cell 4: Demo — Full AI Pipeline

Chạy toàn bộ pipeline trên câu input mẫu:
```
NLP (NER) → ML Cuisine → KB → Filter → Bayes → Rank
```

In [ ]:
# ============================================================
# DEMO: Full Pipeline
# ============================================================
test_input = "I want to cook something with beef and noodle for $10, I have stomachache"

print("=" * 70)
print("FULL AI PIPELINE DEMO")
print("=" * 70)
print(f"\n📝 Input: \"{test_input}\"")

# --- STEP 1: NLP (NER + ML Naive Bayes) ---
print("\n" + "-" * 50)
print("STEP 1: NLP Parser (NER → ML Naive Bayes)")
print("-" * 50)
parsed = nlp_parser.parse(test_input)
print(f"  Budget:      ${parsed.budget}")
print(f"  Ingredients: {parsed.ingredients}")
print(f"  Health:      {parsed.health_conditions}")
print(f"  Cuisine:     {parsed.target_cuisine} (method={parsed.cuisine_method}, conf={parsed.cuisine_confidence:.4f})")
print(f"  Confidence:  {parsed.confidence:.2f}")

# --- STEP 2: Knowledge Base (Forward Chaining) ---
print("\n" + "-" * 50)
print("STEP 2: Knowledge Base (Forward Chaining)")
print("-" * 50)
kb_result = kb_engine.infer(parsed.health_conditions)
print(f"  Fired rules:   {kb_result.get('fired_rules', [])}")
print(f"  Excluded ings:  {kb_result.get('excluded_ingredients', [])}")
print(f"  Excluded tags:  {kb_result.get('excluded_tags', [])}")
if kb_result.get('warnings'):
    for w in kb_result['warnings']:
        print(f"  ⚠️  {w}")

# --- STEP 3: Recipe Filtering ---
print("\n" + "-" * 50)
print("STEP 3: Recipe Filtering")
print("-" * 50)
excluded_ings = set(kb_result.get('excluded_ingredients', []))
excluded_tags = set(kb_result.get('excluded_tags', []))

filtered_recipes = []
for r in recipes:
    r_ings = {ing.get('name', '').lower() for ing in r.get('ingredients', [])}
    r_tags = {t.lower() for t in r.get('tags', [])}
    if not r_ings.intersection(excluded_ings) and not r_tags.intersection(excluded_tags):
        filtered_recipes.append(r)
print(f"  Before: {len(recipes)} recipes")
print(f"  After:  {len(filtered_recipes)} recipes")

# --- STEP 4: Bayesian Risk Assessment ---
print("\n" + "-" * 50)
print("STEP 4: Bayesian Risk (Extended Noisy-OR + Synergy)")
print("-" * 50)
evaluated = []
for r in filtered_recipes[:10]:
    result = bayes_eval.evaluate(r, user_conditions=parsed.health_conditions)
    evaluated.append((r, result))

# Sort by preference_score (Expected Utility)
evaluated.sort(key=lambda x: x[1].preference_score, reverse=True)

print(f"  Top 3 recipes (sorted by P(Like|Evidence)):")
for i, (r, ev) in enumerate(evaluated[:3], 1):
    print(f"  {i}. {r.get('name','N/A'):30s}  P(Like)={ev.preference_score:.3f}  Risk={ev.risk_score:.3f}  Level={ev.risk_level}")

# --- STEP 5: ML Prediction Detail ---
print("\n" + "-" * 50)
print("STEP 5: ML Naive Bayes — Top-3 Cuisine Predictions")
print("-" * 50)
top_k = nlp_parser.ml_classifier.predict_top_k(parsed.ingredients, k=3)
for cuisine, conf in top_k:
    bar = '█' * int(conf * 40)
    print(f"  {cuisine:15s} {bar} {conf:.2%}")

print("\n" + "=" * 70)
print("✅ Pipeline hoàn tất!")
print("=" * 70)

---
## Cell 5: Kiểm thử từng Module

In [ ]:
# ============================================================
# TEST 1: ML Classifier — Multinomial Naive Bayes
# ============================================================
print("=" * 60)
print("TEST 1: Multinomial Naive Bayes Cuisine Classifier")
print("=" * 60)

classifier = CuisineNaiveBayesClassifier()
classifier.train(recipes)
print(f"  Trained: {len(classifier.class_counts)} cuisines, {len(classifier.vocab)} vocab terms")

test_ings = [
    (["olive oil", "tomato sauce", "pasta"], "Italian"),
    (["salmon", "sushi rice", "wasabi"], "Japanese"),
    (["beef", "fish sauce", "lemongrass"], "Vietnamese"),
]
print()
for ings, expected in test_ings:
    pred, conf = classifier.predict_with_confidence(ings)
    match = "✅" if pred and expected.lower() in pred.lower() else "❌"
    print(f"  {match} {str(ings):55s} → {str(pred):15s} ({conf:.4f})")

In [ ]:
# ============================================================
# TEST 2: CSP Solver — Dual-constraint Pruning
# ============================================================
print("\n" + "=" * 60)
print("TEST 2: CSP Solver (Forward Checking + Bounds Propagation)")
print("=" * 60)

if recipes:
    test_recipe = recipes[0]
    csp_result = csp_solver.solve(
        recipe=test_recipe,
        ingredients_db=ingredients_db,
        budget=10.0,
        calorie_range=(200, 800)
    )
    print(f"  Recipe:     {test_recipe.get('name', 'N/A')}")
    print(f"  Success:    {csp_result.get('success', False)}")
    if csp_result.get('success'):
        print(f"  Cost:       ${csp_result.get('total_cost', 0):.2f}")
        print(f"  Calories:   {csp_result.get('total_calories', 0):.0f}")
        print(f"  Backtracks: {csp_result.get('backtracks', 0)}")

In [ ]:
# ============================================================
# TEST 3: A* Search — Canonical State Representation
# ============================================================
print("\n" + "=" * 60)
print("TEST 3: A* Meal Planner (Canonical State Space)")
print("=" * 60)

search_result = search_engine.search(
    candidate_recipes=recipes[:8],
    budget=20.0,
    days=3,
    calorie_range_per_day=(200, 800)
)
print(f"  Success:       {search_result.success}")
print(f"  Nodes explored: {search_result.nodes_explored}")
if search_result.success:
    print(f"  Total cost:    ${search_result.total_cost:.2f}")
    print(f"  Total calories: {search_result.total_calories:.0f}")
    for i, r in enumerate(search_result.planned_recipes, 1):
        print(f"    Day {i}: {r.get('name','N/A')} (${r.get('cost',0):.2f}, {r.get('calories',0)} cal)")

In [ ]:
# ============================================================
# TEST 4: Bayesian Risk — Extended Noisy-OR + Synergy
# ============================================================
print("\n" + "=" * 60)
print("TEST 4: Extended Noisy-OR with Synergy Hidden Causes")
print("=" * 60)

spicy_recipe = {
    "name": "Bún bò Huế",
    "ingredients": [
        {"name": "beef", "quantity": 200},
        {"name": "spicy", "quantity": 10},
        {"name": "ớt", "quantity": 5},
        {"name": "sa tế", "quantity": 15},
    ],
    "tags": ["spicy", "Vietnamese"]
}
risk_score, risk_factors = bayes_eval.assess_risk(
    spicy_recipe,
    health_conditions=["đau dạ dày"]
)
print(f"  Recipe:  {spicy_recipe['name']}")
print(f"  P(Risk) = {risk_score:.4f}  (naturally ∈ [0,1])")
print(f"  Factors:")
for rf in risk_factors:
    print(f"    • {rf['yếu_tố']:20s} P={rf['xác_suất']}  — {rf['mô_tả']}")

# Verify no synergy = lower risk
mild_recipe = {
    "name": "Canh rau",
    "ingredients": [{"name": "spinach", "quantity": 100}],
    "tags": ["healthy"]
}
risk_mild, _ = bayes_eval.assess_risk(mild_recipe)
print(f"\n  Comparison: Bún bò Huế risk={risk_score:.4f} vs Canh rau risk={risk_mild:.4f}")

print("\n✅ All module tests passed.")

---
## Cell 6: Giao diện tương tác

Chạy nhiều test cases tự động.

In [ ]:
# ============================================================
# Multi-query Demo
# ============================================================

def analyze(user_input):
    """Chạy pipeline AI rút gọn."""
    parsed = nlp_parser.parse(user_input)
    kb_result = kb_engine.infer(parsed.health_conditions)
    excluded = set(kb_result.get('excluded_ingredients', []))
    filtered = [
        r for r in recipes
        if not {ig.get('name','').lower() for ig in r.get('ingredients',[])}.intersection(excluded)
    ]

    print(f"📝 \"{user_input}\"")
    print(f"   Cuisine: {parsed.target_cuisine} ({parsed.cuisine_method}, {parsed.cuisine_confidence:.2%})")
    print(f"   Ings: {parsed.ingredients}  |  Health: {parsed.health_conditions}")
    print(f"   Available: {len(filtered)}/{len(recipes)} recipes")

    # ML top predictions
    top = nlp_parser.ml_classifier.predict_top_k(parsed.ingredients, k=3)
    preds = ", ".join([f"{c}({p:.0%})" for c, p in top])
    print(f"   ML predictions: {preds}")
    print()


queries = [
    "I want pasta with olive oil and tomato sauce for $8",
    "Sushi with salmon and rice, budget $15",
    "Something with beef and chili, I have diabetes",
    "Chicken soup with ginger, no spicy, I have gout",
    "Tofu and mushroom stir fry, vegetarian",
]

print("=" * 70)
print("MULTI-QUERY ANALYSIS")
print("=" * 70 + "\n")
for q in queries:
    analyze(q)
print("✅ Done.")

---
## Cell 7: TF-IDF Similarity Search

In [ ]:
# ============================================================
# TF-IDF Cosine Similarity Search
# ============================================================
print("=" * 60)
print("TF-IDF Cosine Similarity Search")
print("=" * 60)

queries = ["pasta cheese Italian", "sushi salmon Japanese", "spicy beef noodle soup"]

for query in queries:
    matches = extractor.find_similar(query, top_k=3)
    print(f"\n🔎 Query: '{query}'")
    for m in matches:
        print(f"   {m.rank}. {m.recipe.get('name', 'N/A')} (similarity={m.similarity:.3f})")

print("\n✅ Complete.")